In [ ]:
import torch
import pandas as pd

In [ ]:
heart = pd.read_csv('heart.csv')
print(heart.shape)
print(heart.columns)
print(heart)

(918, 12)
Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS',
       'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope',
       'HeartDisease'],
      dtype='object')
     Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  \
0     40   M           ATA        140          289          0     Normal   
1     49   F           NAP        160          180          0     Normal   
2     37   M           ATA        130          283          0         ST   
3     48   F           ASY        138          214          0     Normal   
4     54   M           NAP        150          195          0     Normal   
..   ...  ..           ...        ...          ...        ...        ...   
913   45   M            TA        110          264          0     Normal   
914   68   M           ASY        144          193          1     Normal   
915   57   M           ASY        130          131          0     Normal   
916   57   F           ATA        

In [ ]:
heart_encoded = pd.get_dummies(heart, columns=['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope'], drop_first=True)
heart_data = torch.tensor(heart_encoded.astype(float).to_numpy(), dtype=torch.float32)
print(heart_data)
print(heart_data.shape)

tensor([[ 40., 140., 289.,  ...,   0.,   0.,   1.],
        [ 49., 160., 180.,  ...,   0.,   1.,   0.],
        [ 37., 130., 283.,  ...,   0.,   0.,   1.],
        ...,
        [ 57., 130., 131.,  ...,   1.,   1.,   0.],
        [ 57., 130., 236.,  ...,   0.,   1.,   0.],
        [ 38., 138., 175.,  ...,   0.,   0.,   1.]])
torch.Size([918, 16])


Check the cleanness of the data

In [ ]:
print('NAN?', torch.isnan(heart_data).any())
print('inf?', torch.isinf(heart_data).any())

NAN? tensor(False)
inf? tensor(False)


Let's do some distribution analysis

In [ ]:
print(f'Mean: {torch.mean(heart_data)}')
print(f'Min: {torch.min(heart_data)}')
print(f'Max: {torch.max(heart_data)}')
print(f'Std: {torch.max(heart_data)}')

Mean: 32.910648345947266
Min: -2.5999999046325684
Max: 603.0
Std: 603.0


Now let's split the data into features and target variable

In [ ]:
X = heart_data[:, :-1]
Y = heart_data[:, -1]
print(X.shape)
print(Y.shape)

torch.Size([918, 15])
torch.Size([918])


Now let's split the dataset into training and test

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.2, random_state=42)

In [ ]:
print(f"X_train: {X_train.shape} X_test: {X_test.shape}")
print(f"Y_train; {Y_train.shape} Y_test: {Y_test.shape}")

X_train: torch.Size([734, 15]) X_test: torch.Size([184, 15])
Y_train; torch.Size([734]) Y_test: torch.Size([184])


In [ ]:
import torch.nn as nn

class LinearRegressionModel(nn.Module):
  def __init__(self,in_features, out_features):
    super().__init__()
    self.linear_layer = nn.Linear(in_features, out_features)
    self.sigmoid = nn.Sigmoid()

  def forward(self,x):
    x = self.linear_layer(x)
    x = self.sigmoid(x)
    return x

# Initiating the model
model = LinearRegressionModel(in_features=15,out_features=1)
print(model)



LinearRegressionModel(
  (linear_layer): Linear(in_features=15, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


Now let's select our **optimizer** and ***loss function***

In [ ]:
import torch.optim as optim

learning_rate = 0.01

optimizer = optim.Adam(model.parameters(), lr = learning_rate)
# Binary Cross Entropy for  classification
loss_fun = nn.BCELoss()

In [ ]:
epochs = 1000

for epoch in range(epochs):
  y_hat = model(X_train)
  # Loss calcualtion
  loss = loss_fun(y_hat, Y_train.unsqueeze(1))

  # The three line mantra
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if epoch % 100 == 0:
    print(f'Epoch: {epoch}  Loss: {loss.item():.4f}')

Epoch: 0  Loss: 0.1253
Epoch: 100  Loss: 0.1217
Epoch: 200  Loss: 0.1188
Epoch: 300  Loss: 0.1165
Epoch: 400  Loss: 0.1146
Epoch: 500  Loss: 0.1130
Epoch: 600  Loss: 0.1117
Epoch: 700  Loss: 0.1106
Epoch: 800  Loss: 0.1096
Epoch: 900  Loss: 0.1088


#Evaluation

In [ ]:
model.eval()
with torch.inference_mode():
  y_pred = model(X_test)
  test_loss = loss_fun(y_pred, Y_test.unsqueeze(1))

print(f'Test Loss: {test_loss.item():.4f}')

Test Loss: 0.1217


In [ ]:
from sklearn.metrics import accuracy_score

y_pred_class = (y_pred >=0.5).float()

accuracy = accuracy_score(Y_test.cpu().numpy(), y_pred_class.cpu().numpy())
print(f"the model's accuacy is: {accuracy * 100:.2f}%" )

the model's accuacy is: 95.11%


### How to infer the model for real-life use

To use the model in a real-life scenario, you would provide it with new data that has the same structure and preprocessing applied as your training data. The model will output a probability, which you then convert to a binary prediction (0 or 1) using a chosen threshold (e.g., 0.5).

In [ ]:
# Example of inference with new data
# Create a dummy new data point with 15 features, ensuring it's a tensor of float32
# Replace these values with actual new patient data after appropriate preprocessing
new_data_point = torch.tensor([45., 120., 200., 0., 150., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0.], dtype=torch.float32).unsqueeze(0) # unsqueeze(0) to add batch dimension

model.eval() # Set the model to evaluation mode
with torch.inference_mode():
  new_prediction_prob = model(new_data_point)

# Convert probability to binary prediction
new_prediction_class = (new_prediction_prob >= 0.5).float()

print(f'New data point prediction probability: {new_prediction_prob.item():.4f}')
if new_prediction_class.item() == 1:
  print('Prediction: Heart Disease (1)')
else:
  print('Prediction: No Heart Disease (0)')